In [6]:
import jax
import jax.numpy as jnp
from function_utils import construct_func_from_intensity_matrix
from function_utils import construct_func_from_payment_matrix
from probability_callbacks import ProbabilityCallbacks
from prototype_1 import semimarkov_solver as pt1_mks
from prototype_1_heun import semimarkov_solver as pt1_mks_heun
from prototype_2 import semimarkov_solver as pt2_mks
from prototype_3 import semimarkov_solver as pt3_mks
from prototype_4 import semimarkov_solver as pt4_mks
from prototype_5 import semimarkov_solver as pt5_mks
from prototype_6 import semimarkov_solver as pt6_mks
from prototype_7 import semimarkov_solver as pt7_mks
from prototype_8 import semimarkov_solver as pt8_mks
from functools import partial, reduce

#jax.config.update("jax_enable_x64", True)

In [2]:
@jax.jit
def mu_3_scale(t, u, scale):
    my_poly2 = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.01
    
    my_poly = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.09
    
    return scale * (my_poly + my_poly2)

@jax.jit
def mu_3(t, u):
    my_poly2 = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.01
    
    my_poly = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.09
    
    return my_poly + my_poly2

@jax.jit
def mu_2(t, u):
    return 0.15 + 0 * u

@jax.jit
def mu_2_scale(t, u, scale):
    return scale * 0.15 + 0 * u

In [3]:
def evaluate_functions(matrix, *args, **kwargs):
    output = jax.tree_util.tree_map(
        lambda f: f(*args, **kwargs),
        matrix
    )
    return output

@jax.jit
def unit_payment(t, u, *args, **kwargs):
    return jnp.ones((1, 1))

@jax.jit
def dur_payment(t, u, *args, **kwargs):
    return jnp.exp(-(t + u))

@jax.jit
def mu_3(t, u):
    my_poly2 = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.01
    
    my_poly = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.09
    
    return my_poly + my_poly2

u = jnp.expand_dims(jnp.linspace(0, 30, 30 * 12 + 1, endpoint=True), axis=0)

dB = (unit_payment, {'unit': unit_payment, 'dur': dur_payment}, None, None, None)

dB_j = (
    (None, unit_payment, {'unit': unit_payment, 'dur': dur_payment}, None, None),
    None,
    (None, None, None, None, None),
    None,
    None
)

function_matrix = (
    (None, mu_3, mu_3, mu_3, None),
    (None, None, mu_3, mu_3, mu_3),
    (None, None, None, None, None),
    (None, None, None, None, None),
    (None, None, None, None, None)
)

mu = evaluate_functions(function_matrix, 0, u)
B = evaluate_functions(dB_j, 0, u)

In [ ]:
def compute_jump_payments(mu, payments):
    result_matrix = []

    for m_row, p_row in zip(mu, payments):
        if p_row is None or m_row is None:
            result_matrix.append(None)
            continue
            
        current_row_res = []
        
        for m_val, b_val in zip(m_row, p_row):
            if m_val is None or b_val is None:
                current_row_res.append(None)
            else:
                prod = jax.tree_util.tree_map(lambda x: m_val * x, b_val)
                current_row_res.append(prod)
        
        result_matrix.append(tuple(current_row_res))
        
    return tuple(result_matrix)

def add_with_nones(x, y):
        if x is None: return y
        if y is None: return x
        return x + y

def reduce_state(x):
    row_sums = []
    for row in x:
        leaves = jax.tree_util.tree_leaves(row)
        row_sum = sum(leaves) if leaves else None            
        row_sums.append(row_sum)
        
    return tuple(row_sums)

def reduce_cashflow(x):    
    return reduce(lambda a, b: jax.tree_util.tree_map(add_with_nones, a, b), x)
    

In [35]:
tree = (jnp.array([1, 2]), None, {'a': jnp.array([3]), 'b': None})
leaves = jax.tree_util.tree_leaves(tree)

In [96]:
cj = compute_jump_payments(mu, B)
cj_2 = reduce_state(cj)

test = (None, None, None, None, None)

In [97]:
cj_2

(Array([[1.4261376 , 1.3495317 , 1.2797577 , 1.2160976 , 1.1579144 ,
         1.104644  , 1.0557858 , 1.0108948 , 0.9695752 , 0.93147516,
         0.89628094, 0.8637121 , 0.833519  , 0.8054781 , 0.77938926,
         0.75507367, 0.7323705 , 0.71113586, 0.69123995, 0.6725664 ,
         0.6550104 , 0.63847756, 0.62288237, 0.6081482 , 0.59420526,
         0.5809909 , 0.56844795, 0.556525  , 0.5451751 , 0.53435576,
         0.52402836, 0.51415765, 0.5047118 , 0.4956615 , 0.48698014,
         0.47864342, 0.47062904, 0.46291673, 0.45548773, 0.44832507,
         0.44141293, 0.43473685, 0.4282835 , 0.42204076, 0.41599715,
         0.41014224, 0.40446636, 0.39896053, 0.39361638, 0.38842615,
         0.3833827 , 0.37847936, 0.37370974, 0.3690679 , 0.36454862,
         0.36014658, 0.3558569 , 0.3516751 , 0.34759668, 0.34361792,
         0.33973485, 0.33594388, 0.33224165, 0.32862473, 0.32509038,
         0.32163522, 0.31825686, 0.31495255, 0.31171978, 0.30855593,
         0.3054585 , 0.30242628, 0

In [89]:
collapse_cashflow([test])

(None, None, None, None, None)

In [3]:
intensity_matrix = (
    (None, mu_3, mu_3, mu_3, None),
    (None, None, mu_3, mu_3, mu_3),
    (None, None, None, None, None),
    (None, None, None, None, None),
    (None, None, None, None, None)
)

intensity_matrix_scale = (
    (None, mu_3_scale, mu_3_scale, mu_3_scale, None),
    (None, None, mu_3_scale, mu_3_scale, mu_3_scale),
    (None, None, None, None, None),
    (None, None, None, None, None),
    (None, None, None, None, None)
)

intensity_matrix_dense = (
    (None, mu_3, mu_3, mu_3, mu_3),
    (mu_2, None, mu_3, mu_3, mu_3),
    (mu_2, mu_2, None, mu_3, mu_3),
    (mu_2, mu_2, mu_2, None, mu_3),
    (mu_2, mu_2, mu_2, mu_2, mu_3),
)

intensity_matrix_dense_scale = (
    (None, mu_3_scale, mu_3_scale, mu_3_scale, mu_3_scale),
    (mu_2_scale, None, mu_3_scale, mu_3_scale, mu_3_scale),
    (mu_2_scale, mu_2_scale, None, mu_3_scale, mu_3_scale),
    (mu_2_scale, mu_2_scale, mu_2_scale, None, mu_3_scale),
    (mu_2_scale, mu_2_scale, mu_2_scale, mu_2_scale, None),
)

"""
intensity_matrix = [
    [None, mu_3],
    [None, None]
]
"""

intensity = construct_func_from_intensity_matrix(intensity_matrix)
intensity_scale = construct_func_from_intensity_matrix(intensity_matrix_scale)
intensity_d = construct_func_from_intensity_matrix(intensity_matrix_dense)
intensity_scale_d = construct_func_from_intensity_matrix(intensity_matrix_dense_scale)
step_size = 1 / 24

u = jnp.linspace(0, 30, 30 * 12 + 1, endpoint=True)

u = jnp.expand_dims(u, axis=0)
scale = jnp.expand_dims(jnp.linspace(0.95, 1.05, 100), 1)
intensity_kwargs = {'scale': scale}

In [4]:
# Baseline with Euler scheme
def bench1():
    result = pt1_mks(grid=u,
                intensity=intensity,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# Improved difference version (euler scheme)
def bench2():
    result = pt2_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# With einsum
def bench3():
    result = pt3_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# With Heun scheme
def bench4():
    result = pt4_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# Baseline with heun scheme
def bench5():
    result = pt1_mks_heun(grid=u,
                intensity=intensity,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench6():
    result = pt5_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench7():
    result = pt6_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench8():
    result = pt7_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench9():
    result = pt8_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

#bench1()
#bench2()
#bench3()
#bench4()
#bench5()
#bench6()
bench7()
bench8()
bench9()
1+1

2

In [5]:
# Sparse intensity matrix with batch size 1
#%timeit -r 10 -n 100 bench1() # Baseline Euler
#%timeit -r 10 -n 100 bench2() # New Euler
#%timeit -r 10 -n 100 bench3() # New with einsum Euler
#%timeit -r 10 -n 100 bench4() # New with einsum Heun
#%timeit -r 10 -n 100 bench5() # Baseline Heun
#%timeit -r 10 -n 100 bench6() # New with sparse loop Heun
#%timeit -r 20 -n 100 bench7() # New with sparse loop Heun (modified)
%timeit -r 10 -n 100 bench8() # New with sparse loop Heun vmap
%timeit -r 10 -n 100 bench9() # New with sparse loop Heun vmap + improved update

5.84 ms ± 151 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
6.69 ms ± 47.6 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [5]:
# Baseline with Euler scheme
def bench1_d():
    result = pt1_mks(grid=u,
                intensity=intensity_d,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# Improved difference version (euler scheme)
def bench2_d():
    result = pt2_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_d,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# With einsum
def bench3_d():
    result = pt3_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_d,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# With Heun scheme
def bench4_d():
    result = pt4_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_d,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# Baseline with heun scheme
def bench5_d():
    result = pt1_mks_heun(grid=u,
                intensity=intensity_d,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench6_d():
    result = pt5_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench7_d():
    result = pt6_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench8_d():
    result = pt7_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench9_d():
    result = pt8_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

#bench1_d()
#bench2_d()
#bench3_d()
#bench4_d()
#bench5_d()
#bench6_d()
#bench7_d()
bench8_d()
bench9_d()
1+1

2

In [6]:
# Dense intensity matrix with batch size 1
#%timeit -r 10 -n 100 bench1_d() # Baseline Euler
#%timeit -r 10 -n 100 bench2_d() # New Euler
#%timeit -r 10 -n 100 bench3_d() # New with einsum Euler
#%timeit -r 10 -n 100 bench4_d() # New with einsum Heun
#%timeit -r 10 -n 100 bench5_d() # Baseline Heun
#%timeit -r 10 -n 100 bench6_d() # New with sparse loop Heun
#%timeit -r 10 -n 100 bench7_d() # New with sparse loop Heun (modified)
%timeit -r 10 -n 100 bench8_d() # New with sparse loop Heun vmap
%timeit -r 10 -n 100 bench9_d() # New with sparse loop Heun vmap + improved update

7.4 ms ± 130 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
8.08 ms ± 30.2 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [6]:
def bench1_2():
    result = pt1_mks(grid=u,
                intensity=intensity_scale,
                intensity_kwargs=intensity_kwargs,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench2_2():
    result = pt2_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench3_2():
    result = pt3_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench4_2():
    result = pt4_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench5_2():
    result = pt1_mks_heun(grid=u,
                intensity=intensity_scale,
                intensity_kwargs=intensity_kwargs,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench6_2():
    result = pt5_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench7_2():
    result = pt6_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench8_2():
    result = pt7_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench9_2():
    result = pt8_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()


#bench1_2()
#bench2_2()
#bench3_2()
#bench4_2()
#bench5_2()
#bench6_2()
#bench7_2()
bench8_2()
bench9_2()
2+2

4

In [7]:
# Sparse intensity matrix with batch size 100
#%timeit -r 10 -n 100 bench1_2() # Baseline 
#%timeit -r 10 -n 100 bench2_2() # New Euler
#%timeit -r 10 -n 100 bench3_2() # New with einsum Euler
#%timeit -r 10 -n 100 bench4_2() # New with einsum Heun
#%timeit -r 10 -n 100 bench5_2() # Baseline heun
#%timeit -r 10 -n 100 bench6_2() # New with sparse loop Heun
#%timeit -r 10 -n 100 bench7_2() # New with sparse loop Heun (modified)
%timeit -r 10 -n 100 bench8_2() # New with sparse loop Heun vmap
%timeit -r 10 -n 100 bench9_2() # New with sparse loop Heun vmap + improved update

9.62 ms ± 56.7 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
12.1 ms ± 49.9 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [9]:
def bench1_2_d():
    result = pt1_mks(grid=u,
                intensity=intensity_scale_d,
                intensity_kwargs=intensity_kwargs,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench2_2_d():
    result = pt2_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale_d,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench3_2_d():
    result = pt3_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale_d,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench4_2_d():
    result = pt4_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale_d,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench5_2_d():
    result = pt1_mks_heun(grid=u,
                intensity=intensity_scale_d,
                intensity_kwargs=intensity_kwargs,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench6_2_d():
    result = pt5_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench7_2_d():
    result = pt6_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench8_2_d():
    result = pt7_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench9_2_d():
    result = pt8_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_dense_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

#bench1_2_d()
#bench2_2_d()
#bench3_2_d()
#bench4_2_d()
#bench5_2_d()
#bench6_2_d()
#bench7_2_d()
bench8_2_d()
bench9_2_d()
2+2

4

In [10]:
# Dense intensity matrix with batch size 100
#%timeit -r 10 -n 100 bench1_2_d() # Baseline 
#%timeit -r 10 -n 100 bench2_2_d() # New Euler
#%timeit -r 10 -n 100 bench3_2_d() # New with einsum Euler
#%timeit -r 10 -n 100 bench4_2_d() # New with einsum Heun
#%timeit -r 10 -n 100 bench5_2_d() # Baseline heun
#%timeit -r 10 -n 100 bench6_2_d() # New with sparse loop Heun
#%timeit -r 10 -n 100 bench7_2_d() # New with sparse loop Heun (modified)
%timeit -r 10 -n 100 bench8_2_d() # New with sparse loop Heun vmap
%timeit -r 10 -n 100 bench9_2_d() # New with sparse loop Heun vmap + improved update

15.1 ms ± 47.8 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
42.5 ms ± 23.5 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
